# TASK 2: NEURAL NETWORK (MNIST)

## Part A: Dataset Understanding
* **What is MNIST?** It is a classic dataset in machine learning containing 70,000 grayscale images of handwritten digits.
* **Input Image Dimensions:** Each image is 28x28 pixels.
* **Number of Classes:** 10 classes (digits 0 through 9).
* **Labels:** The labels represent the actual human-verified integer value of the handwritten digit in the image.

## Part B: Why Preprocess?
* **Normalization (Dividing by 255):** Pixel values range from 0 to 255. Neural networks learn faster and more stably when input values are scaled between 0 and 1.
* **Flattening:** A standard dense neural network expects a 1D array of numbers as input. We must flatten the 2D 28x28 image grid into a 1D array of 784 pixels.

## Part D: Activation Functions
* **Why are they used?** Activation functions introduce non-linearity into the network. Without them, no matter how many layers a neural network has, it would just behave like a single linear regression model, unable to learn complex visual patterns.
* **ReLU (Rectified Linear Unit):** Used in hidden layers because it is computationally efficient and helps prevent the "vanishing gradient" problem, allowing the network to learn faster.
* **Softmax:** Used in the output layer for multi-class classification. It converts the network's raw output scores into probabilities that sum to 1, allowing us to confidently predict the most likely digit.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# ==========================================
# PART B: Data Preprocessing
# ==========================================

# 1. Load the MNIST dataset
mnist = tf.keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 2. Normalize pixel values to be between 0 and 1
X_train, X_test = X_train / 255.0, X_test / 255.0

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# ==========================================
# PART C & E: Build and Train the Neural Network
# ==========================================

# 1. Build the baseline model
base_model = Sequential([
    Flatten(input_shape=(28, 28)),          # Input layer (flattens 28x28 to 784)
    Dense(128, activation='relu'),          # Hidden layer with ReLU
    Dense(10, activation='softmax')         # Output layer with Softmax (10 classes)
])

# 2. Compile the model
base_model.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

# 3. Train the model (recording history for plotting)
print("\n--- Training Baseline Model ---")
history = base_model.fit(X_train, y_train, epochs=5, validation_split=0.2)

# 4. Plot Training/Validation Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.show()

# ==========================================
# PART F: Evaluation & Confusion Matrix
# ==========================================

# 1. Evaluate accuracy on unseen test data
test_loss, test_acc = base_model.evaluate(X_test, y_test, verbose=0)
print(f"\nBaseline Model Test Accuracy: {test_acc:.4f}")

# 2. Generate predictions and Confusion Matrix
y_pred_probs = base_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Baseline Model)')
plt.xlabel('Predicted Digit')
plt.ylabel('Actual Digit')
plt.show()

# Print Precision, Recall, and F1-score
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==========================================
# PART G: Experimentation
# ==========================================
# Experiment: Increasing the complexity (adding a hidden layer & more neurons)

exp_model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu'),          # Increased to 256 neurons
    Dense(128, activation='relu'),          # Added a second hidden layer
    Dense(10, activation='softmax')
])

exp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\n--- Training Experimental Model ---")
exp_model.fit(X_train, y_train, epochs=5, validation_split=0.2, verbose=1)

exp_test_loss, exp_test_acc = exp_model.evaluate(X_test, y_test, verbose=0)
print(f"\nExperimental Model Test Accuracy: {exp_test_acc:.4f}")

## PART F: Understanding the Confusion Matrix
The confusion matrix shows exactly where the model gets confused. The diagonal line represents correct predictions. Any numbers outside the diagonal represent misclassifications. For example, looking at the matrix, the model occasionally confuses 4s and 9s, or 3s and 5s, because those handwritten shapes share similar pixel structures.

## PART G: Experimentation Analysis
* **What did I change?** I increased the complexity of the neural network by expanding the first hidden layer from 128 to 256 neurons and adding a second hidden layer with 128 neurons.
* **What changed in the results?** The training accuracy increased slightly, and the network learned complex patterns faster.
* **Why did the change affect performance?** Adding more neurons and layers increases the network's "capacity" to learn intricate details and non-linear boundaries. However, making it too complex on a simple dataset like MNIST can risk overfitting, where the model memorizes the training data but fails to generalize.